In [63]:
from langchain_core.tools import tool
from typing import TypedDict
import os
import langchain_openai
from langgraph.graph import START,END,StateGraph,MessagesState
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_openai import ChatOpenAI
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENROUTERAPIKEY")
import sqlite3
import cursor
from langchain_core.messages import SystemMessage
from langgraph.graph.message import add_messages

In [64]:

pip install cursor

Note: you may need to restart the kernel to use updated packages.


In [65]:
import sqlite3
from langgraph.graph import MessagesState

In [66]:
db_name="expenses.db"

def create_database():
    connection=sqlite3.connect(db_name)
    cursor=connection.cursor()
    
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS "expenses"(
        expense_id INTEGER PRIMARY KEY AUTOINCREMENT,
        amount REAL NOT NULL,
        category TEXT NOT NULL,
        description TEXT,
        date TEXT
        )
        """
    )
    connection.commit()
    connection.close()

In [67]:
def add_sample_data():
    connection=sqlite3.connect(db_name)
    cursor=connection.cursor()
    cursor.execute("""
    SELECT COUNT(*) FROM expenses
    """)
    count=cursor.fetchone()[0]

    if count==0:
        sample_data=[
            (500,"food","lunch","23-05-24"),
            (400,"food","dinner","24-05-24"),
            (700,"entertainment","sports","24-07-24"),
            (150,"transport","uber","27-07-24"),
            (1200,"shopping","ahlia","29-07-24")
        ]
        cursor.executemany(
            """
            INSERT INTO expenses
            (amount,category,description,date)
            VALUES(?,?,?,?)
            """,sample_data
        )

    connection.commit()
    connection.close()

create_database()
add_sample_data()

In [68]:
def get_expenses_using_category(category:str):
    connection=sqlite3.connect(db_name)
    cursor=connection.cursor()

    cursor.execute(
        """
        SELECT * FROM expenses WHERE category=?
        """,(category,)
    )
    results=cursor.fetchall()
    connection.close()
    return results

In [69]:
def add_expenses_into_database(amount:float,category:str,description:str,date:str):
    connection=sqlite3.connect(db_name)
    cursor=connection.cursor()

    cursor.execute(
        """
        INSERT INTO expenses(amount,category,description,date)
        VALUES(?,?,?,?)
        """,(amount,category,description,date)
    )
    connection.commit()
    connection.close()
    return "Added successfully"

In [70]:
def delete_expenses_using_id(expense_id:int):
    connection=sqlite3.connect(db_name)
    cursor=connection.cursor()

    cursor.execute(
        """
        DELETE FROM expenses WHERE expense_id=?
        """,(expense_id,)
    )
    connection.commit()
    connection.close()
    return "Deleted Successfully"

In [71]:
def get_total_expenses(category:str|None=None):
    connection=sqlite3.connect(db_name)
    cursor=connection.cursor()

    if category:
        cursor.execute(
            """
            SELECT SUM(amount) FROM expenses WHERE category=?
            """,(category,)
        )

    else:
        cursor.execute(
            """
            SELECT category,SUM(amount) FROM expenses GROUP BY category
            """
        )

    results=cursor.fetchall()
    connection.close()
    return results

In [72]:
@tool
def get_expense(category:str):
    """
    Get all expenses belonging to a specific category.

    Use this tool when the user asks to see expenses
    from a particular category such as food, transport,
    shopping, etc.
    """
    expense=get_expenses_using_category(category)

    if not expense:
        return f"No expense found for {category}"

    results=[]
    for row in expense:
        expense_id,amount,category,description,date=row
        results.append(
            f"ID:{expense_id}",
            f"Amount:{amount}",
            f"Category:{category}",
            f"Description:{description}",
            f"Date:{date}")
    return "\n".join(results)


@tool
def add_expense(amount:float,category:str,description:str,date:str):
    """
    Use this tool to add expenses to database as per users information
    """

    add_expenses_into_database(amount,category,description,date)
    return "Added the expenses succesfully"

@tool
def delete_expenses(expenses_id:int):
    """
    Delete expenses using provided id number as per users request
    """
    delete_expenses_using_id(expenses_id)
    return f"Deleted Successfully {expenses_id}"


@tool
def expense_summary(category:str|None=None):
    """
    Provide expense summary as per users request,if category is provided it will return specific category expense,otherwise it will return all expenses of each category.
    """
    results=get_total_expenses(category)
    return results

tools=[get_expense,add_expense,delete_expenses,expense_summary]

In [73]:
llm=ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=1000,
    model="openrouter/free",
    api_key=secret_value_0
)

llm_with_tools=llm.bind_tools(tools)

In [74]:
class AgentState(MessagesState):
    pass

In [75]:
def call_llm(state:AgentState):
    system_message=SystemMessage(
        content="""
        You are an AI expense assistant.

        You can help the user manage their expenses.

        Available actions:
        - Get expenses by category
        - Add expenses
        - Delete expenses
        - Get expense summaries

        Use tools whenever the user's request requires
        information from or modification of the database.

        Do not invent database information.

        After receiving a tool result, explain the result
        clearly to the user.
        """
    )
    messages=[system_message]+state['messages']
    response=llm_with_tools.invoke(messages)

    return {"messages":[response]}

In [76]:
tool_node=ToolNode(tools)

In [77]:
builder=StateGraph(AgentState)
builder.add_node("llm",call_llm)
builder.add_node("tools",tool_node)

In [78]:
builder.add_edge(START,"llm")
builder.add_conditional_edges("llm",tools_condition)
builder.add_edge("tools","llm")

In [79]:
graph=builder.compile()

In [81]:
result=graph.invoke({
    "messages":["user","i want to see my total food expenses"]
})

print(result["messages"][-1].content)

Your total food expenses amount to $900.00.
